# Lesson 02
Run lessons 1–1 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第2课 均值回归与统计套利

**Quant Research and Execution Lab · 02_mean_reversion_and_stat_arb · 完整课件 · 2026-10-13**

本课作为第二模块紧接第一课，代码与产物保留在当前合并IPYNB。02.1—02.10可按顺序独立运行；CPU足够。依赖刚更新但内存仍是旧版本时，使用“代码执行程序 → 重启会话并运行所有单元”。本课只读取2008—2022开发数据；不发布、不连接券商、不导出本地文件。

## 本课目标与研究假设

由单资产z-score进入两资产相对价值策略，判断价差偏离是否会回归，并把双腿交易费、借券、现金与融资写进同一个账本。预先固定IWM/SPY候选：它们具有共同的美国股票风险敞口，但小盘与更广泛股票市场的风格差异可能使关系长期改变。这个经济联系只提供候选理由，不保证协整或盈利。

均值回归认为偏离某个稳定水平后，未来变化倾向于回到该水平。统计套利是用统计关系提出交易假设，仍承担估计误差、结构变化、流动性和尾部风险。本课仅为固定ETF配对研究，不等同于全市场统计套利。

## 术语与符号

|术语|含义与本课用法|
|---|---|
|相关性|两组变动同步程度；共同市场上涨可产生高相关，不保证残差回归。|
|平稳性|分布的统计性质随时间较稳定；有限样本检验不能证明永远平稳。|
|I(1)与协整|价格水平非平稳、差分更接近平稳；某个线性组合平稳时可能有协整关系。|
|OLS|只用过去窗口估计截距alpha与系数beta，最小化残差平方和。|
|价差与z-score|本课价差是对数回归残差u；z是相对于过去残差均值和标准差的偏离。|
|做多／做空|正持仓受益于上涨，负持仓受益于下跌；空头需借券、可能被召回。|
|净／总敞口|净敞口为有符号金额之和；总敞口为绝对金额之和，净额小不代表风险小。|
|半衰期|局部AR(1)估计下偏离缩小一半的交易日数；不是未来盈利时间承诺。|

A=IWM，B=SPY；$P_{A,t},P_{B,t}$为一致口径收盘价；$\alpha_t,\beta_t$为过去样本估计参数；$u_t$为残差，$\mu_t,\sigma_t$为估计窗口残差均值与标准差；$s_t$为收盘状态，$w_{i,t}$为开盘目标权重，$q_i$为模型单位，$B$为现金，$E$为账户权益。

$$u_t=\log P_{A,t}-\alpha_t-\beta_t\log P_{B,t},\qquad z_t=(u_t-\mu_t)/\sigma_t$$
$$u_{j+1}=a+\phi u_j+\varepsilon_{j+1},\qquad h=-\log(2)/\log(\phi),\quad 0<\phi<1$$

先用过去252个交易日估计，每21日重估。当前收盘不进入本次拟合窗口，但可用于本日z值。ADF原假设是单位根；Engle–Granger原假设是不协整。普通残差ADF p值只做诊断，协整判定使用coint校正p值。I(1)诊断条件与0.05阈值是本课预设启发式，不构成完整单位根证明；反复滚动检验不能当作多次独立证据。

## 决策、成交与停止规则

|状态|预设行为|
|---|---|
|空仓且合格|$2\le |z|<4$时入场；正z做空A做多B，负z反向。|
|持仓且价差回归|进入中心带或直接越过均值时退出，不能因跳过中心带继续反向持有。|
|关系不合格|beta非正、半衰期无效、启用过滤后不满足协整／I(1)诊断时退出。|
|进一步恶化|向不利方向到达4个标准差时止损，或持仓40个交易日后退出。|
|不可借|执行时若任一目标空头不可借，整个组合归零并按逐腿成本关闭；恢复可借后由当时指令决定。|

周一收盘可得信号 → 周二开盘成交 → 周三开盘估值。退出也按下一交易日开盘执行，所以价格跳空可能使实际损失超出z止损。不可借分支是执行层覆盖，不会伪造历史借券数据。基线假设可借；2011-08到2012-07不可借区间只是显式压力假设。

## 对冲金额与共同账本

给定权益E、总敞口比例g和状态s：

$$w_A=gs/(1+|\beta|),\qquad w_B=-\beta w_A,\qquad q_i=w_iE_{after}/O_i$$

beta描述对数变动的近似金额对冲关系，不是两腿股数比；它也不保证美元中性或市场beta中性。借券、融资和风险暴露仍须单独计算。

$$E_{pre}=B+\sum_i q_iO_i,\qquad F_i=|\Delta q_i|O_i(c_{commission}+c_{slippage})$$
$$B'=B-\sum_i\Delta q_iO_i-\sum_iF_i-F_{borrow}-F_{funding}$$
$$E_{end}=B'+\sum_iq'_iO_{i,next},\qquad r=E_{end}/E_{pre}-1$$

算法同时求解扣交易费后的权益与目标持仓，避免把成本扣两次。借券按各腿空头开盘市值×假设年费率×日历天数/365计提，周末也计；担保现金=空头市值×1.5，现金不足担保部分按假设融资利率计费。担保只是现金分类，不能再次从账户权益减去。模型假定该融资额度可用，不实现真实券商保证金追缴和强平；总敞口有配置上限。

## 数据与记账口径

Yahoo教学下载保留供应商Open/Close、Adj Close、调整因子、分红拆股事件和时间字段，记录版本、覆盖、获取时点与SHA256。双腿只取共同交易日；不靠未来值填缺失，失败时停止。Yahoo非股息调整价可能已按拆股处理，不等于交易所原始逐笔报价。公开发布前需核对数据许可。

本账本用调整价与模型单位，复用第一课自融资恒等式的多资产扩展，并用单资产解析参考与第一课引擎交叉核验。公司行动不能再重复入账。真实空头股息补偿、支付日期、税、整数股、冲击和真实借券限制仍需原始价格账本验证；本课输出不能称为可执行套利。

## 实验路线与产物

|单元格|内容与完成证据|
|---|---|
|02.1|固定版本、种子、所有研究参数与运行日志。|
|02.2|两腿统一价格表、事件、数据清单和可得时间。|
|02.3|滚动OLS、协整／ADF、半衰期、状态机及冻结模型。|
|02.4|共用多资产账本、逐腿费用、现金担保、净／总敞口和信号接口。|
|02.5|费用、借券、融资、不可借、损益守恒、因果性和非法行情检查。|
|02.6|单资产、无对冲、固定对冲、滚动对冲、等权再平衡、冻结模型及图表。|
|02.7|一次改一因素的成本／借券／不可借／过滤对照与真实退出案例。|
|02.8|可运行双腿手算，与扣费后目标求解对照。|
|02.9|固定种子的结构变化合成案例，明确区别于历史行情与业绩。|
|02.10|练习参考、实际结果差异、全部最终验收和完整实验台账。|

主对照fixed只固定初始beta，其他信号相同，用于隔离对冲方式变化；frozen_model冻结alpha、beta和残差分布，是额外的整套模型对照。equal_weight是双腿等权日频再平衡，不称为纯粹一次买入持有。所有模型用相同本金、预热、执行起点和估值终点；每个规则使用相同账本。

## 手算与练习参考

本金10000，总敞口8000，beta=1.5：A金额3200、B金额−4800；A价100、B价200时，模型单位32和−24。1bp佣金加4bp滑点，总交易费用4。现金=10000−3200+4800−4=11596，权益=现金+3200−4800=9996。短卖收到的现金不是利润。A次日103、B次日201，市场损益=32×3−24×1=72；周五到周一借券费=4800×3%×3/365，再从账户扣除。02.8运行给出精确答案。固定份额手算与引擎的扣费后权重求解略有差异，课件同时展示两种口径。

1. 基础实验：全部运行，核对手算、信号延迟、每腿费用和账本。参考答案在02.5与02.8的断言和输出中。
2. 个人对照：选择成本、借券或过滤一项，先写机制假设，再报告02.7实际指标、订单和敞口，不预先宣称改善。参考差异由02.10读取本次结果计算。
3. 调优问题：什么机制使价差回归？半衰期是否短到覆盖费用？相关性是否只来自共同市场？多个候选中选显著配对是否造成选择偏差？答案必须结合开发证据、完整试验记录和后续新数据，不用“p小于0.05”替代收益验证。
4. 进阶：OU、Kalman或因子残差任选一项，先登记新增参数和否决条件，沿用相同目标接口和账本；本课不以实现所有进阶模型作为通过标准。

## 失败条件与验收

价差可能不回归、结构变化可能暂时被滚动估计吸收、借券可能召回、滑点与融资可能吞噬收益、调整数据可能被修订。过滤后无交易、模型亏损、成本敏感和关系断裂都应保留。结构变化示例是机制演示，不能充当真实回测证据。

标准课件验收：干净会话自上而下运行；全部10个代码单元格有实际输出；未来改写不改变过去；逐腿损益减全部费用等于账户权益变化；所有模型窗口一致；不可借时无持仓；手算正确；价格、价差、净值、回撤、净／总敞口、交易记录与失败案例均可查；最终日志状态为completed_and_verified。个人自主实验仍需学习者独立完成，课件提供任务和参考。

## 官方资料与研究入口

- [statsmodels coint：原假设与I(1)条件](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html)
- [statsmodels ADF：单位根诊断](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html)
- [yfinance下载与使用说明](https://github.com/ranaroussi/yfinance)
- [Backtesting.py：订单、成本与成交时间参考](https://kernc.github.io/backtesting.py/doc/backtesting/backtesting.html)

**本次标准基线未入场的处理**：严格过滤下的零交易保留为否决结果。02.6另给相同未过滤指令下的无对冲／固定／滚动对冲诊断；02.7的diagnostic情景再隔离成本、借券和不可借因素。它们都明确标为过滤消融，不能借此宣称满足协整或得到有效套利。

所有代码在下方展开；最终输出保留开发实验与模拟案例的明确标注。


In [ ]:
# 02.1 环境、预设候选与实验登记（可从此处独立运行）
%pip -q install numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.3 yfinance==0.2.66 statsmodels==0.14.5
import yfinance as yf
import json, hashlib, warnings, sys
from datetime import datetime, timezone
from importlib.metadata import version
for package in ["numpy", "pandas", "matplotlib", "statsmodels"]:
    loaded=sys.modules.get(package)
    if loaded is not None and getattr(loaded,"__version__",version(package)) != version(package):
        raise RuntimeError("核心依赖已更新：使用 重启会话并运行所有单元，再继续。")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from IPython.display import display
L2_CONFIG={"assets":["IWM","SPY"],"start":"2008-01-01","end":"2023-01-01",
    "window":252,"refit_every":21,"entry_z":2.,"exit_z":.5,"stop_z":4.,
    "max_holding":40,"gross_target":.8,"commission_bps":1.,"slippage_bps":4.,
    "borrow_annual":.03,"funding_annual":.05,"short_collateral":1.5,
    "initial_cash":10000.,"p_threshold":.05,"adf_maxlag":4,"single_window":20,
    "max_gross":2.,"seed":42,
    "borrow_status":"assumed availability, not historical observations"}
np.random.seed(42)
L2_LOG={"started_utc":datetime.now(timezone.utc).isoformat(),"python":sys.version,
    "versions":{p:version(p) for p in ["numpy","pandas","matplotlib","yfinance","statsmodels"]},
    "status":"started","accounting":"adjusted-price synthetic units"}
L2_EXPERIMENTS=[]
print(json.dumps(L2_CONFIG,ensure_ascii=False,indent=2))
print(L2_LOG["versions"])


## 02A 两腿数据与关系估计

02.2先建立一致价格表并保存事件与来源。02.3在每个决策日前的252行拟合OLS：alpha是截距，beta是金额对冲比的近似；用同一过去窗口的残差分布标准化当前z值。fit_last_session必须早于信号日。检验失败、半衰期无效或beta非正时，合格基线不入场。

固定对冲fixed只固定最初beta，整套冻结模型另称frozen_model。后者不能通过未来数据重新估计参数。

In [ ]:
# 02.2 两腿数据与事件；不使用后期样本，不自动填补行情
L2_RAW={}; L2_MANIFEST=[]
try:
    for asset in L2_CONFIG["assets"]:
        d=yf.download(asset,start=L2_CONFIG["start"],end=L2_CONFIG["end"],
            auto_adjust=False,actions=True,multi_level_index=False,progress=False,threads=False)
        if d is None or d.empty: raise RuntimeError(f"{asset}下载失败；停止，不伪造数据")
        if isinstance(d.columns,pd.MultiIndex): d.columns=d.columns.get_level_values(0)
        d.index=pd.DatetimeIndex(d.index).tz_localize(None).normalize()
        d=d.sort_index()
        if d.index.has_duplicates: raise ValueError(f"{asset}重复交易日")
        cols=["Open","Close","Adj Close"]
        d[cols]=d[cols].apply(pd.to_numeric,errors="coerce")
        if d[cols].isna().any().any() or (d[cols]<=0).any().any(): raise ValueError("价格缺失/非正")
        L2_RAW[asset]=d
        L2_MANIFEST.append({"instrument":asset,"source":"Yahoo/yfinance",
            "first":str(d.index.min().date()),"last":str(d.index.max().date()),"rows":len(d),"retrieved_utc":datetime.now(timezone.utc).isoformat(),
            "price_basis":"vendor OHLC and dividend-adjusted OHLC retained",
            "usage":"personal educational research; do not assume redistribution rights",
            "events_available":[x for x in ["Dividends","Stock Splits"] if x in d],
            "sha256":hashlib.sha256(d.to_csv(float_format="%.10g").encode()).hexdigest()})
except Exception as exc:
    L2_LOG.update(status="data_failed",failure=str(exc)); print(L2_LOG); raise
l2_open=pd.concat({a:d.Open*d["Adj Close"]/d.Close for a,d in L2_RAW.items()},axis=1)
l2_close=pd.concat({a:d["Adj Close"] for a,d in L2_RAW.items()},axis=1)
l2_common=l2_open.dropna().index.intersection(l2_close.dropna().index)
l2_dropped=len(l2_open.index)-len(l2_common)
l2_open=l2_open.loc[l2_common]; l2_close=l2_close.loc[l2_common]
assert len(l2_open)>500
l2_logprice=np.log(l2_close)
print("只保留双腿都有价格的交易日；丢弃不共同的行数：",l2_dropped)
display(pd.DataFrame(L2_MANIFEST),l2_close.head())
L2_PRICES=pd.concat({a:pd.DataFrame({"vendor_open":d.Open,"vendor_close":d.Close,
    "adjustment_factor":d["Adj Close"]/d.Close,"adjusted_open":d.Open*d["Adj Close"]/d.Close,
    "adjusted_close":d["Adj Close"],"dividend":d.get("Dividends",0.),"split":d.get("Stock Splits",0.)})
    for a,d in L2_RAW.items()},names=["instrument","session"]).reset_index()
L2_PRICES["available_at"]=(pd.to_datetime(L2_PRICES.session)+pd.Timedelta(hours=16)).dt.tz_localize("America/New_York")
print("原始供应商口径、调整口径、公司行动与时间字段：")
display(L2_PRICES.head(6))
print("复权已处理公司行动，不在模型单位账本重复记分红/拆股；事件仍保存在L2_RAW中。")


In [ ]:
# 02.3 滚动OLS、Engle-Granger、ADF与半衰期；只用t之前252行估计
# IWM为A，SPY为B；普通残差ADF只作诊断，协整采用coint的校正p值。
def l2_fit_features(logprice,window=None,refit_every=None):
    window=L2_CONFIG["window"] if window is None else window
    refit_every=L2_CONFIG["refit_every"] if refit_every is None else refit_every
    if window<30 or refit_every<1 or len(logprice)<=window:
        raise ValueError("估计窗口/重估频率/样本长度无效")
    rows=[]; estimate=None
    for t in range(window,len(logprice)):
        if (t-window)%refit_every==0:
            past=logprice.iloc[t-window:t]; y=past.iloc[:,0].to_numpy(); x=past.iloc[:,1].to_numpy()
            alpha,beta=np.linalg.lstsq(np.c_[np.ones(len(x)),x],y,rcond=None)[0]
            residual=y-alpha-beta*x
            phi=np.linalg.lstsq(np.c_[np.ones(len(residual)-1),residual[:-1]],residual[1:],rcond=None)[0][1]
            half=-np.log(2)/np.log(phi) if 0<phi<1 else np.nan
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                p_coint=coint(y,x,trend="c",maxlag=L2_CONFIG["adf_maxlag"],autolag="aic")[1]
                p_adf=adfuller(residual,maxlag=L2_CONFIG["adf_maxlag"],autolag="AIC")[1]
                level_p=[adfuller(v,maxlag=L2_CONFIG["adf_maxlag"],autolag="AIC")[1] for v in [y,x]]
                diff_p=[adfuller(np.diff(v),maxlag=L2_CONFIG["adf_maxlag"],autolag="AIC")[1] for v in [y,x]]
            estimate={"alpha":alpha,"beta":beta,"mean":residual.mean(),"std":residual.std(ddof=1),
                "coint_p":p_coint,"residual_adf_p":p_adf,"half_life":half,
                "i1_diagnostic":min(level_p)>L2_CONFIG["p_threshold"] and max(diff_p)<L2_CONFIG["p_threshold"],
                "fit_last_session":past.index[-1]}
        current=logprice.iloc[t]
        spread=current.iloc[0]-estimate["alpha"]-estimate["beta"]*current.iloc[1]
        z=(spread-estimate["mean"])/estimate["std"] if estimate["std"]>1e-10 else np.nan
        rows.append({"session":logprice.index[t],**estimate,"spread":spread,"z":z})
    return pd.DataFrame(rows).set_index("session")
l2_features=l2_fit_features(l2_logprice)
# 状态机：不在同一日退出后重新入场；显式保留退出原因和最长持仓。
def l2_states(features,entry=None,exit_z=None,stop=None,max_holding=None,require_coint=True):
    entry=L2_CONFIG["entry_z"] if entry is None else entry
    exit_z=L2_CONFIG["exit_z"] if exit_z is None else exit_z
    stop=L2_CONFIG["stop_z"] if stop is None else stop
    max_holding=L2_CONFIG["max_holding"] if max_holding is None else max_holding
    if not 0<=exit_z<entry<stop or max_holding<1: raise ValueError("状态机参数无效")
    state=0; age=0; rows=[]
    for date,r in features.iterrows():
        quality=(np.isfinite(r.z) and r.beta>0 and np.isfinite(r.half_life) and
                 (not require_coint or (r.coint_p<L2_CONFIG["p_threshold"] and bool(r.i1_diagnostic))))
        reason="hold"; old=state
        if state:
            age+=1
            if not quality: state=0; reason="relationship_failed"
            elif state*r.z>=-exit_z: state=0; reason="mean_reversion_exit"
            elif state*r.z<=-stop: state=0; reason="z_stop"
            elif age>=max_holding: state=0; reason="max_holding_exit"
        elif quality and entry<=abs(r.z)<stop:
            state=-int(np.sign(r.z)); age=0; reason="entry"
        rows.append({"session":date,"state":state,"age":age,"reason":reason,"changed":state!=old})
    return pd.DataFrame(rows).set_index("session")
l2_state=l2_states(l2_features)
l2_fixed_beta=float(l2_features.beta.iloc[0])
# 单资产z-score用于入门对照，均值/标准差来自上一日结束的20日窗口。
l2_prior=l2_close.iloc[:,0].shift(1)
l2_single_z=(l2_close.iloc[:,0]-l2_prior.rolling(L2_CONFIG["single_window"]).mean())/l2_prior.rolling(L2_CONFIG["single_window"]).std()
display(l2_features.head(),l2_state.loc[l2_state.changed].head(10))
print("固定beta（仅首个估计窗口）：",l2_fixed_beta)

# 冻结整个价差模型的附加对照：alpha/beta/均值/标准差只来自首个252日。
# 只固定beta的主对照仍复用滚动信号，故能隔离“对冲比率”这一个因素。
l2_frozen_features=l2_features.copy()
l2_initial_estimate=l2_features.iloc[0]
for field in ["alpha","beta","mean","std","coint_p","residual_adf_p","half_life","i1_diagnostic","fit_last_session"]:
    l2_frozen_features[field]=l2_initial_estimate[field]
l2_frozen_features["spread"]=(l2_logprice.iloc[:,0]-l2_initial_estimate.alpha-
    l2_initial_estimate.beta*l2_logprice.iloc[:,1]).reindex(l2_features.index)
l2_frozen_features["z"]=(l2_frozen_features.spread-l2_initial_estimate["mean"])/l2_initial_estimate["std"]
print("所有阈值、窗口和重估频率已接入L2_CONFIG；冻结模型只用第一个估计窗口。")


## 02B 指令转成持仓与现金账本

02.4把收盘状态移到下一开盘，用目标金额／开盘价换算两腿模型单位。每腿成交费、借券费与市场损益都单列。卖空收到现金，同时产生负资产市值；受限担保现金仍属于账户，不能再次扣减权益。融资只针对担保不足或借入现金部分。

这里的融资额度与借券可得性是情景假设。基线总敞口0.8，单资产测试允许预设1.5融资情景；本课不模拟真实券商强平。

In [ ]:
# 02.4 统一多资产账本：第一课自融资方程的多腿扩展
# 所有第二课模型复用此引擎；费用按每腿成交额，不用“价差涨幅”充当账户收益。
def l2_simulate_multi(opens,target,commission=None,slippage=None,borrow_rate=None,
                      funding=None,available=None,initial=None,collateral=None):
    commission=L2_CONFIG["commission_bps"] if commission is None else commission
    slippage=L2_CONFIG["slippage_bps"] if slippage is None else slippage
    borrow_rate=L2_CONFIG["borrow_annual"] if borrow_rate is None else borrow_rate
    funding=L2_CONFIG["funding_annual"] if funding is None else funding
    initial=L2_CONFIG["initial_cash"] if initial is None else initial
    collateral=L2_CONFIG["short_collateral"] if collateral is None else collateral
    if initial<=0 or collateral<1: raise ValueError("本金/担保比例无效")
    if not opens.columns.equals(target.columns): raise ValueError("价格与权重资产列必须同名同序")
    if not opens.index.is_monotonic_increasing or opens.index.has_duplicates:
        raise ValueError("价格索引需递增且唯一")
    if not target.index.equals(opens.loc[target.index[0]:target.index[-1]].index):
        raise ValueError("权重必须覆盖连续的共同交易日，不能跳过资金费用")
    if commission<0 or slippage<0 or borrow_rate<0 or funding<0: raise ValueError("费率不得为负")
    if target.isna().any().any() or (target.abs().sum(axis=1)>L2_CONFIG["max_gross"]).any(): raise ValueError("目标缺失/总敞口超限")
    p=opens.reindex(target.index); nxt=opens.shift(-1).reindex(target.index)
    ends=pd.Series(opens.index,index=opens.index).shift(-1).reindex(target.index)
    if (not np.isfinite(p.to_numpy()).all() or not np.isfinite(nxt.to_numpy()).all()
        or (p<=0).any().any() or (nxt<=0).any().any()):
        raise ValueError("执行和估值价格必须有限且为正")
    if available is None: available=pd.DataFrame(True,index=target.index,columns=target.columns)
    available=available.reindex(index=target.index,columns=target.columns)
    if available.isna().any().any(): raise ValueError("借券可得性缺失，不能默认为可借")
    cash=float(initial); units=np.zeros(len(target.columns)); rows=[]; trades=[]
    rate=(commission+slippage)/10000
    for date,w in target.iterrows():
        price=p.loc[date].to_numpy(float); future=nxt.loc[date].to_numpy(float)
        weights=w.to_numpy(float); blocked=bool(((weights<0)&~available.loc[date].to_numpy(bool)).any())
        if blocked: weights=np.zeros_like(weights)  # 不可借则关闭整个组合，已有空头买回。
        before=cash+units@price
        if before<=0: raise RuntimeError("权益非正，停止策略，不继续产生无效权重")
        old_value=units*price; post=before
        # 同时求解扣费后权益与各腿目标；成本在账户内只扣一次。
        for _ in range(40):
            delta=weights*post-old_value
            new_post=before-np.abs(delta).sum()*rate
            if abs(new_post-post)<1e-9: post=new_post; break
            post=new_post
        else: raise RuntimeError("持仓/费用求解未收敛")
        delta=weights*post-old_value
        delta[np.abs(delta)<1e-8]=0
        commission_cash=np.abs(delta)*commission/10000
        slippage_cash=np.abs(delta)*slippage/10000
        fees=commission_cash+slippage_cash
        units=units+delta/price; cash=cash-delta.sum()-fees.sum()
        after=cash+units@price
        assert np.isclose(after,before-fees.sum(),atol=1e-6)
        short_value=np.maximum(-units*price,0)
        restricted=collateral*short_value.sum()
        days=(ends.loc[date]-date).days
        borrow=short_value*borrow_rate*days/365
        finance=max(restricted-cash,0)*funding*days/365
        cash-=borrow.sum()+finance
        market_pnl=units*(future-price)
        end=cash+units@future
        assert np.isclose(end-before,market_pnl.sum()-fees.sum()-borrow.sum()-finance,atol=1e-6)
        if end<=0: raise RuntimeError("组合发生权益耗尽，停止并保留失败原因")
        row={"session":date,"end_session":ends.loc[date],"equity_before":before,
            "equity_after_trade":after,"cash":cash,"cash_restricted":restricted,
            "cash_free":cash-restricted,"equity_end":end,"net_return":end/before-1,
            "trade_notional":np.abs(delta).sum(),"transaction_cost":fees.sum(),
            "borrow_cost":borrow.sum(),"funding_cost":finance,"turnover":np.abs(delta).sum()/before,
            "gross_exposure":np.abs(units*price).sum()/after,"net_exposure":(units*price).sum()/after,
            "blocked_by_borrow":blocked,"market_pnl":market_pnl.sum(),
            "net_pnl":end-before,"long_value":np.maximum(units*price,0).sum(),
            "short_value":short_value.sum(),"holding":bool(np.abs(units).sum()>1e-8)}
        for j,a in enumerate(target.columns):
            row[f"units_{a}"]=units[j]; row[f"borrow_{a}"]=borrow[j]
            row[f"market_pnl_{a}"]=market_pnl[j]; row[f"position_value_{a}"]=units[j]*price[j]
            row[f"commission_{a}"]=commission_cash[j]; row[f"slippage_{a}"]=slippage_cash[j]
            if abs(delta[j])>1e-8:
                trades.append({"session":date,"instrument":a,"delta_model_units":delta[j]/price[j],
                    "notional":abs(delta[j]),"commission":commission_cash[j],"slippage":slippage_cash[j],
                    "model_price":price[j],"action":"buy/cover" if delta[j]>0 else "sell/short"})
        assert np.isclose(end,cash+units@future,atol=1e-6)
        rows.append(row)
    ledger=pd.DataFrame(rows).set_index("session")
    ledger["wealth"]=ledger.equity_end/initial
    ledger["drawdown"]=ledger.wealth/ledger.wealth.cummax().clip(lower=1)-1
    return ledger,pd.DataFrame(trades)

# 共同目标接口，available_at不晚于执行日；回归beta仅近似金额对冲，不等于股数比。
def l2_targets(model="rolling",require_coint=True):
    source_features=l2_frozen_features if model=="frozen_model" else l2_features
    state=l2_states(source_features,require_coint=require_coint).state.reindex(l2_open.index).fillna(0)
    beta=l2_features.beta.reindex(l2_open.index).fillna(0)
    w=pd.DataFrame(0.,index=l2_open.index,columns=l2_open.columns)
    if model in ["rolling","fixed","frozen_model"]:
        b=beta if model=="rolling" else l2_fixed_beta
        w.iloc[:,0]=L2_CONFIG["gross_target"]*state/(1+np.abs(b))
        w.iloc[:,1]=-b*w.iloc[:,0]
    elif model=="unhedged": w.iloc[:,0]=L2_CONFIG["gross_target"]*state
    elif model=="single_z":
        f=pd.DataFrame({"z":l2_single_z,"beta":1.,"half_life":1.},index=l2_open.index)
        w.iloc[:,0]=L2_CONFIG["gross_target"]*l2_states(f,require_coint=False).state
    elif model=="equal_weight": w[:]=L2_CONFIG["gross_target"]/len(w.columns)
    else: raise ValueError("未知模型")
    return w.shift(1).iloc[L2_CONFIG["window"]+1:-1].copy()
l2_target=l2_targets()
l2_interface=l2_target.stack().rename("target_weight").reset_index()
l2_interface.columns=["execution_session","instrument","target_weight"]
l2_prev=pd.Series(l2_open.index,index=l2_open.index).shift(1)
l2_interface["available_at"]=pd.to_datetime(l2_interface.execution_session.map(l2_prev))+pd.Timedelta(hours=16)
l2_interface["available_at"]=l2_interface.available_at.dt.tz_localize("America/New_York")
l2_interface["execution_at"]=(pd.to_datetime(l2_interface.execution_session)+pd.Timedelta(hours=9,minutes=30)).dt.tz_localize("America/New_York")
l2_interface["model_id"]="rolling_pair"; l2_interface["horizon"]="next_open"
l2_interface["signal"]=l2_interface.execution_session.map(l2_prev).map(l2_states(l2_features).state)
assert (l2_interface.available_at<l2_interface.execution_at).all()
display(l2_interface.head())


In [ ]:
# 02.5 关键测试：双腿费用、借券/融资、不可借、守恒与时间因果性
l2_test_dates=pd.date_range("2020-01-06",periods=3,freq="B")
l2_test_open=pd.DataFrame({"A":[100.,102.,101.],"B":[200.,202.,201.]},index=l2_test_dates)
l2_test_w=pd.DataFrame([[.4,-.4],[0.,0.]],index=l2_test_dates[:2],columns=["A","B"])
l2_t,l2_tt=l2_simulate_multi(l2_test_open,l2_test_w)
assert (l2_t.transaction_cost>=0).all() and (l2_t.borrow_cost>=0).all()
assert np.isclose(l2_tt.commission.sum()+l2_tt.slippage.sum(),l2_t.transaction_cost.sum())
assert np.allclose(l2_t.equity_end,l2_t.cash+l2_t.units_A*l2_test_open.A.shift(-1).loc[l2_t.index]+l2_t.units_B*l2_test_open.B.shift(-1).loc[l2_t.index])
assert np.allclose((1+l2_t.net_return).cumprod(),l2_t.wealth)
l2_free,_=l2_simulate_multi(l2_test_open,l2_test_w,0,0,0,0)
assert l2_t.equity_end.iloc[-1]<l2_free.equity_end.iloc[-1]
l2_unavailable=pd.DataFrame(False,index=l2_test_w.index,columns=l2_test_w.columns)
l2_blocked,l2_bt=l2_simulate_multi(l2_test_open,l2_test_w,available=l2_unavailable)
assert np.allclose(l2_blocked.equity_end,10000) and l2_bt.empty
# 融资分支用明确的假设杠杆例子验证，实际研究基线总敞口仍为0.8。
l2_constant=pd.DataFrame({"A":[100.,100.,100.]},index=l2_test_dates)
l2_leveraged=pd.DataFrame({"A":[1.5,0.]},index=l2_test_dates[:2])
l2_finance,_=l2_simulate_multi(l2_constant,l2_leveraged,0,0,0,.05,initial=1000)
assert np.isclose(l2_finance.funding_cost.iloc[0],500*.05/365)
# 与第一课引擎交叉核验0/1单资产：相同输入必须得到相同权益。
if "simulate" in globals():
    l2_binary=pd.Series([1.,0.],index=l2_test_dates[:2])
    l2_old=simulate(l2_test_open.A,l2_binary,5,1000)
    l2_new,_=l2_simulate_multi(l2_test_open[["A"]],l2_binary.to_frame("A"),1,4,0,0,initial=1000)
    assert np.allclose(l2_old.equity_end,l2_new.equity_end)
# 改写未来的两腿价格；过去拟合、信号和状态必须保持不变。
l2_cut=len(l2_logprice)//2
l2_changed=l2_logprice.copy(); l2_changed.iloc[l2_cut:]+=np.array([.5,-.2])
l2_changed_features=l2_fit_features(l2_changed.iloc[:l2_cut+25])
l2_before=l2_logprice.index[l2_cut]
pd.testing.assert_frame_equal(l2_features.loc[l2_features.index<l2_before],
                              l2_changed_features.loc[l2_changed_features.index<l2_before])
pd.testing.assert_frame_equal(l2_states(l2_features).loc[:l2_before].iloc[:-1],
                              l2_states(l2_changed_features).loc[:l2_before].iloc[:-1])
assert (l2_features.fit_last_session<l2_features.index).all()
l2_jump=pd.DataFrame({"z":[2.5,-2.],"beta":1.,"half_life":1.},index=l2_test_dates[:2])
l2_jump_state=l2_states(l2_jump,require_coint=False)
assert list(l2_jump_state.state)==[-1,0] and l2_jump_state.reason.iloc[-1]=="mean_reversion_exit"
print("L2 PASS：费用、借券、融资、不可借关闭、账本守恒、复利、单资产兼容与因果性")

# 额外核验：每腿市场PnL减去所有费用，严格等于账户权益变化。
assert np.allclose(l2_t.net_pnl,l2_t.market_pnl-l2_t.transaction_cost-l2_t.borrow_cost-l2_t.funding_cost)
assert np.allclose(l2_t.market_pnl,l2_t.market_pnl_A+l2_t.market_pnl_B)
assert np.allclose(l2_blocked.gross_exposure,0)
# 非法报价/总敞口在交易前拒绝，不能产生看似有效结果。
for bad_price in [0.,np.nan,np.inf]:
    invalid=l2_test_open.copy(); invalid.iloc[0,0]=bad_price
    try: l2_simulate_multi(invalid,l2_test_w)
    except ValueError: pass
    else: raise AssertionError("非法行情未被拒绝")
# 单资产满仓/现金的解析参考，第二课独立运行也必须执行。
reference_cash=1000/(100*1.0005)*102*(1-.0005)
reference_target=pd.DataFrame({"A":[1.,0.]},index=l2_test_dates[:2])
reference_ledger,_=l2_simulate_multi(l2_test_open[["A"]],reference_target,1,4,0,0,initial=1000)
assert np.isclose(reference_ledger.equity_end.iloc[-1],reference_cash)
print("L2 CORE CHECKS COMPLETE：含逐腿PnL归因、非法报价拒绝与独立解析参考")


## 02C 基线与诊断实验

02.6的主要比较使用相同协整过滤；若均空仓，结果说明候选没有满足预设规则。附加未过滤对冲比较明确标为诊断，隔离对冲比率的作用。02.7的严格情景保留零结果，活跃诊断则以no_coint_filter为共同参考，一次只改变费用、借券与可得性。

图表包括价差水平、z值、beta、半衰期、净值、回撤和净／总市场敞口。逐腿订单数与完整往返交易次数不同。

In [ ]:
# 02.6 无对冲/固定/滚动/单资产/简单基准，全部共用账本
L2_LEDGERS={}; L2_TRADES={}
for model in ["rolling","fixed","unhedged","single_z","equal_weight","frozen_model"]:
    w=l2_targets(model)
    L2_LEDGERS[model],L2_TRADES[model]=l2_simulate_multi(l2_open,w)
    assert w.index.equals(l2_target.index)
def l2_metrics(x):
    r=x.net_return; vol=r.std(ddof=1)
    return {"total_return":x.wealth.iloc[-1]-1,"annual_return":x.wealth.iloc[-1]**(252/len(x))-1,
        "sharpe_rf0":r.mean()/vol*np.sqrt(252) if vol>1e-12 else np.nan,
        "max_drawdown":x.drawdown.min(),"mean_gross":x.gross_exposure.mean(),
        "mean_net":x.net_exposure.mean(),"turnover":x.turnover.sum(),
        "trade_cost":x.transaction_cost.sum(),"borrow_cost":x.borrow_cost.sum(),
        "funding_cost":x.funding_cost.sum(),"borrow_block_days":int(x.blocked_by_borrow.sum()),
        "annual_vol":vol*np.sqrt(252),"holding_fraction":x.holding.mean(),
        "min_cash_free":x.cash_free.min(),"market_pnl":x.market_pnl.sum(),"net_pnl":x.net_pnl.sum()}
L2_SUMMARY=pd.DataFrame({k:l2_metrics(v) for k,v in L2_LEDGERS.items()}).T
l2_fmt={x:"{:.2%}" for x in ["total_return","annual_return","annual_vol","max_drawdown","mean_gross","mean_net","holding_fraction"]}
l2_fmt.update({x:"{:.2f}" for x in ["sharpe_rf0","turnover","trade_cost","borrow_cost","funding_cost"]})
display(L2_SUMMARY.style.format(l2_fmt))
display(L2_TRADES["rolling"].tail(12))
display(L2_LEDGERS["rolling"].tail(5))
print("滚动策略逐腿订单数：",len(L2_TRADES["rolling"]))
print("主对照fixed只固定金额对冲比，冻结整个价差的附加对照叫frozen_model。")
print("通过I(1)诊断与协整阈值的收盘占比：",((l2_features.coint_p<L2_CONFIG["p_threshold"])&l2_features.i1_diagnostic).mean())
# 简单基准为双腿等权日频再平衡，佣金滑点也照记，不冒称纯粹一次买入持有。
fig,axes=plt.subplots(3,1,figsize=(13,10))
axes[0].plot(l2_features.index,l2_features.z,label="Rolling residual z")
for v in [-4,-2,-.5,.5,2,4]: axes[0].axhline(v,color="gray",alpha=.4,lw=.7)
axes[0].set(title="Residual z-score: entry, exit and failure thresholds")
for k,x in L2_LEDGERS.items():
    axes[1].plot(pd.DatetimeIndex([x.index[0],*x.end_session.tolist()]),np.r_[1.,x.wealth.to_numpy()],label=k)
axes[1].set(title="Same window and account engine: net wealth",ylabel="Wealth")
for k in ["rolling","fixed","unhedged"]:
    x=L2_LEDGERS[k]; axes[2].plot(x.end_session,100*x.drawdown,label=k)
axes[2].set(title="Hedge comparison: drawdowns",ylabel="Drawdown (%)")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
axes[0].plot(l2_features.index,l2_features.coint_p,label="coint p"); axes[0].axhline(.05,color="red")
axes[1].plot(l2_features.index,l2_features.beta,label="rolling beta")
axes[1].plot(l2_features.index,l2_features.half_life,label="local half-life (sessions)",alpha=.6)
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

# 价差和市场净/总敞口是课件必需产物，不只展示z值与净值。
fig,axes=plt.subplots(4,1,figsize=(13,11),sharex=True)
axes[0].plot(l2_features.index,l2_features.spread,label="Rolling log residual")
axes[0].plot(l2_frozen_features.index,l2_frozen_features.spread,label="Frozen-model log residual",alpha=.7)
axes[0].set(title="Spread levels: do not treat these as directly tradable prices")
axes[1].plot(l2_features.index,l2_features.beta,label="Rolling amount hedge beta")
axes[1].axhline(l2_fixed_beta,color="gray",label="Initial fixed beta")
axes[2].plot(l2_features.index,l2_features.half_life,label="Local AR(1) half-life")
axes[2].set(ylabel="Trading sessions")
l2_base=L2_LEDGERS["rolling"]
axes[3].plot(l2_base.index,l2_base.gross_exposure,label="Gross exposure")
axes[3].plot(l2_base.index,l2_base.net_exposure,label="Net exposure")
axes[3].set(ylabel="Fraction of equity",xlabel="Session")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
# 所有基线登记到同一试验台账，使用实际运行指标，不遗漏负结果。
for model,x in L2_LEDGERS.items():
    L2_EXPERIMENTS.append({"id":"model_"+model,"kind":"baseline_comparison",
        "config":L2_CONFIG.copy(),"metrics":l2_metrics(x),
        "data_hashes":[d["sha256"] for d in L2_MANIFEST]})

# 补充诊断：基线过滤未入场时，三种对冲在同一未过滤指令下比较。
# 这是已登记的过滤消融，不改变合格基线，不声称满足协整条件或可交易。
L2_HEDGE_DIAGNOSTIC={}; L2_HEDGE_TRADES={}
for model in ["rolling","fixed","unhedged"]:
    diagnostic_w=l2_targets(model,require_coint=False)
    dx,dt=l2_simulate_multi(l2_open,diagnostic_w)
    L2_HEDGE_DIAGNOSTIC[model]=dx; L2_HEDGE_TRADES[model]=dt
    L2_EXPERIMENTS.append({"id":"diagnostic_hedge_"+model,"kind":"unfiltered_hedge_ablation",
        "coint_filter":False,"metrics":l2_metrics(dx),"config":L2_CONFIG.copy(),
        "data_hashes":[d["sha256"] for d in L2_MANIFEST]})
print("未过滤信号的对冲诊断：这些结果不构成协整套利证据。")
L2_HEDGE_TABLE=pd.DataFrame({k:l2_metrics(v) for k,v in L2_HEDGE_DIAGNOSTIC.items()}).T
display(L2_HEDGE_TABLE.style.format(l2_fmt))
print("未过滤滚动对冲的真实逐腿交易例子：")
display(L2_HEDGE_TRADES["rolling"].head(10))
fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
for k,x in L2_HEDGE_DIAGNOSTIC.items():
    axes[0].plot(pd.DatetimeIndex([x.index[0],*x.end_session.tolist()]),np.r_[1.,x.wealth.to_numpy()],label=k)
    axes[1].plot(x.end_session,100*x.drawdown,label=k)
axes[0].set(title="Diagnostic only: hedge comparison without cointegration filter",ylabel="Wealth")
axes[1].set(ylabel="Drawdown (%)")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()


In [ ]:
# 02.7 一次改变一个因素：交易费、借券、不可借及取消协整过滤
l2_scenarios={"baseline":{},"cost20":{"slippage":20.-L2_CONFIG["commission_bps"]},"borrow15pct":{"borrow_rate":.15},
              "unavailable_stress":{},"no_coint_filter":{},
    "diagnostic_cost20":{"slippage":20.-L2_CONFIG["commission_bps"]},
    "diagnostic_borrow15pct":{"borrow_rate":.15},"diagnostic_unavailable":{},
    "diagnostic_no_cost":{"commission":0.,"slippage":0.,"borrow_rate":0.,"funding":0.}}
l2_assumed_available=pd.DataFrame(True,index=l2_target.index,columns=l2_target.columns)
l2_stress_available=l2_assumed_available.copy()
# 明确标注的假设不可借区间，不是历史借券事件。
l2_stress_available.loc["2011-08-01":"2012-07-31",:]=False
L2_COMPARISONS={}; l2_results=[]
for name,kw in l2_scenarios.items():
    filtered=(name!="no_coint_filter" and not name.startswith("diagnostic_"))
    w=l2_targets(require_coint=filtered)
    availability=l2_stress_available if name in ["unavailable_stress","diagnostic_unavailable"] else l2_assumed_available
    x,tr=l2_simulate_multi(l2_open,w,available=availability,**kw)
    L2_COMPARISONS[name]=x
    l2_results.append({"scenario":name,"leg_orders":len(tr),**l2_metrics(x)})
    L2_EXPERIMENTS.append({"id":name,"pair":L2_CONFIG["assets"],"scope":"2008-2022 development",
        "metrics":l2_metrics(x),"config":L2_CONFIG.copy(),
        "overrides":kw,"coint_filter":filtered,
        "borrow_availability":"synthetic unavailable interval" if name=="unavailable_stress" else "assumed true",
        "data_hashes":[d["sha256"] for d in L2_MANIFEST]})
L2_COMPARISON_TABLE=pd.DataFrame(l2_results).set_index("scenario")
display(L2_COMPARISON_TABLE.style.format(l2_fmt))
fig,ax=plt.subplots(figsize=(13,4))
for name,x in L2_COMPARISONS.items(): ax.plot(x.end_session,x.wealth,label=name)
ax.set(title="Predeclared development experiments; borrowing is assumed",ylabel="Net wealth")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
l2_failures=l2_features.join(l2_state).loc[lambda d:d.reason.isin(["relationship_failed","z_stop","max_holding_exit"])]
print("失效/退出案例：")
display(l2_failures[["z","beta","coint_p","residual_adf_p","half_life","reason"]].head(12))
print("参考解答：相关性不保证协整；普通残差ADF p值不能替代协整校正p值。")
print("beta是近似金额对冲比：先分配总敞口，再按各腿价格转成模型单位。")
print("两腿交易费和空头借券分别计费；周末也计借券日历天；限制现金可能产生融资。")
print("无交易/低收益/过滤失败都是有效研究结果，不放宽规则凑业绩。")
print("作业：任选一个对照，提交预先假设、实际指标、机制解释和否决证据。")
L2_LOG.update(status="core_complete_waiting_for_handcalc_and_review",finished_utc=datetime.now(timezone.utc).isoformat(),
    checks="fees, borrow, funding, unavailable, conservation, compound, single_asset_compatibility, causality, timing passed",
    data_manifest=L2_MANIFEST,assumptions=L2_CONFIG,experiments=L2_EXPERIMENTS)
print(json.dumps(L2_LOG,ensure_ascii=False,indent=2))
# 不将开发实验称独立样本外，不据这份模型单位账本声称可执行套利。

print("真实开发实验自动解读：")
l2_baseline_metrics=L2_COMPARISON_TABLE.loc["baseline"]
for name,r in L2_COMPARISON_TABLE.iterrows():
    print(f"{name}: 累计收益{r.total_return:.2%}，夏普{r.sharpe_rf0:.2f}，最大回撤{r.max_drawdown:.2%}，"
          f"逐腿订单{r.leg_orders:.0f}，相对基线累计收益变化{r.total_return-l2_baseline_metrics.total_return:+.2%}")
if L2_TRADES["rolling"].empty:
    print("过滤后的预设候选未交易：这是当前规则与数据下的否决结果，不放宽阈值补业绩。")
print("上述差异限于开发期；同一pair中滚动检验不等于多次独立统计证据。")

print("活跃交易的费用诊断：固定未过滤信号，以no_coint_filter为共同参考。")
diagnostic_names=["no_coint_filter","diagnostic_cost20","diagnostic_borrow15pct",
    "diagnostic_unavailable","diagnostic_no_cost"]
active_cost_table=L2_COMPARISON_TABLE.loc[diagnostic_names].copy()
active_cost_table["delta_return_vs_same_signal"]=active_cost_table.total_return-L2_COMPARISON_TABLE.loc["no_coint_filter","total_return"]
display(active_cost_table.style.format({**l2_fmt,"delta_return_vs_same_signal":"{:+.2%}"}))
print("活跃诊断的成本差异只解释当前规则和模型费用；它不把未通过的配对升级为合格套利。")


## 02D 手算与结构变化

02.8用假设的固定32／−24单位核对双腿金额、现金、三天借券和净收益，再对照扣费后目标权重的共用引擎。固定份额与目标权重再平衡是两种不同指令，略有数值差异应能解释。

02.9使用已标注的合成数据：模拟前段AR(1)均值回归，后段关系持续偏移。即使两腿收益相关性仍高，残差也可能不再回归；过滤可能滞后。它验证机制，不证明真实套利表现。

In [ ]:
# 02.8 可运行双腿手算：明确标注假设价格，不当作真实行情
# A做多32单位，B做空24单位；固定金额3200/-4800，总敞口8000。
# 周五开盘入场，周一开盘估值；借券按3个日历日而非1个交易日计算。
manual_initial=10000.; manual_beta=1.5; manual_gross=8000.
manual_price=np.array([100.,200.]); manual_next=np.array([103.,201.])
manual_dollars=np.array([manual_gross/(1+manual_beta),-manual_beta*manual_gross/(1+manual_beta)])
manual_units=manual_dollars/manual_price
manual_commission=np.abs(manual_dollars)*.0001
manual_slippage=np.abs(manual_dollars)*.0004
manual_fee=(manual_commission+manual_slippage).sum()
manual_cash=manual_initial-manual_dollars.sum()-manual_fee
manual_short=-np.minimum(manual_dollars,0)
manual_borrow=manual_short*.03*3/365
manual_restricted=manual_short.sum()*1.5
manual_funding=max(manual_restricted-manual_cash,0)*.05*3/365
manual_pnl=manual_units*(manual_next-manual_price)
manual_cash_end=manual_cash-manual_borrow.sum()-manual_funding
manual_end=manual_cash_end+manual_units@manual_next
manual_table=pd.DataFrame({"instrument":["Synthetic A","Synthetic B"],
    "entry_price":manual_price,"signed_amount":manual_dollars,"units":manual_units,
    "commission":manual_commission,"slippage":manual_slippage,"borrow_3days":manual_borrow,
    "price_pnl":manual_pnl})
display(manual_table)
print(f"成交费{manual_fee:.2f}；买卖后现金{manual_cash:.2f}；担保现金{manual_restricted:.2f}")
print(f"双腿市场损益{manual_pnl.sum():.2f}；借券{manual_borrow.sum():.6f}；融资{manual_funding:.6f}")
print(f"周一权益{manual_end:.6f}；净收益{manual_end/manual_initial-1:.4%}")
assert np.allclose(manual_units,[32.,-24.]) and np.isclose(manual_fee,4.)
assert np.isclose(manual_end,manual_initial+manual_pnl.sum()-manual_fee-manual_borrow.sum()-manual_funding)
# 共用引擎按扣费后的权益求目标权重，故与固定32/-24手算略有差异；不是记账误差。
manual_dates=pd.to_datetime(["2020-01-10","2020-01-13","2020-01-14"])
manual_opens=pd.DataFrame([[100.,200.],[103.,201.],[103.,201.]],index=manual_dates,columns=["A","B"])
manual_targets=pd.DataFrame([[.32,-.48],[0.,0.]],index=manual_dates[:2],columns=["A","B"])
manual_ledger,manual_trades=l2_simulate_multi(manual_opens,manual_targets,
    commission=1,slippage=4,borrow_rate=.03,funding=.05,initial=10000,collateral=1.5)
exact_post_equity=manual_initial/(1+.8*.0005)
assert np.isclose(manual_ledger.equity_after_trade.iloc[0],exact_post_equity)
assert np.isclose(manual_ledger.units_A.iloc[0],.32*exact_post_equity/100)
assert np.isclose(manual_ledger.units_B.iloc[0],-.48*exact_post_equity/200)
print("引擎的扣费后目标与入场/平仓订单（第二行还要计两腿退出费）：")
display(manual_ledger[["equity_before","equity_after_trade","cash","cash_restricted",    "borrow_cost","funding_cost","market_pnl","equity_end"]],manual_trades)
L2_EXPERIMENTS.append({"id":"manual_two_leg","kind":"synthetic_accounting_check",    "days":3,"fixed_units_equity":float(manual_end),"target_weight_entry_equity":float(manual_ledger.equity_end.iloc[0])})
print("L2 HAND CALC PASS：双腿仓位、现金、逐腿费用、周末借券及扣费后权重")


In [ ]:
# 02.9 合成结构变化案例：机制验证，不声称历史套利收益
# 固定种子；前段价差AR(1)，后段加入持续偏移，检验关系变坏时是否停止。
rng=np.random.default_rng(L2_CONFIG["seed"])
n=750; synthetic_dates=pd.bdate_range("2010-01-04",periods=n)
synthetic_x=4.+np.cumsum(.001+rng.normal(0,.008,n))
synthetic_u=np.zeros(n)
for k in range(1,n): synthetic_u[k]=.90*synthetic_u[k-1]+rng.normal(0,.007)
synthetic_y=.2+1.1*synthetic_x+synthetic_u
synthetic_break_at=450
synthetic_broken_y=synthetic_y.copy()
synthetic_broken_y[synthetic_break_at:]+=.004*np.arange(n-synthetic_break_at)
synthetic_clean=pd.DataFrame({"A":synthetic_y,"B":synthetic_x},index=synthetic_dates)
synthetic_broken=pd.DataFrame({"A":synthetic_broken_y,"B":synthetic_x},index=synthetic_dates)
synthetic_features=l2_fit_features(synthetic_broken)
synthetic_states=l2_states(synthetic_features)
synthetic_reference_half=-np.log(2)/np.log(.90)
print(f"已知模拟AR系数0.90，理论半衰期{synthetic_reference_half:.2f}交易日。")
print("此案例只校验机制：滚动估计可能暂时吸收变化，过滤不会保证立刻识别断裂。")
synthetic_diagnostics=pd.DataFrame({"return_correlation":[synthetic_clean.diff().iloc[1:].corr().iloc[0,1],    synthetic_broken.diff().iloc[1:].corr().iloc[0,1]],    "sample_type":["synthetic stable relationship","synthetic structural break"]})
display(synthetic_diagnostics)
fig,axes=plt.subplots(3,1,figsize=(13,8),sharex=True)
known_residual=synthetic_broken.A-.2-1.1*synthetic_broken.B
axes[0].plot(synthetic_dates,known_residual,label="Known synthetic residual")
axes[0].set(title="Synthetic mechanism example: relationship break",ylabel="Residual")
axes[1].plot(synthetic_features.index,synthetic_features.z,label="Rolling z")
axes[1].axhline(L2_CONFIG["stop_z"],color="gray"); axes[1].axhline(-L2_CONFIG["stop_z"],color="gray")
axes[2].plot(synthetic_states.index,synthetic_states.state,label="Decision state")
for ax in axes:
    ax.axvline(synthetic_dates[synthetic_break_at],color="red",ls="--",label="Known break")
    ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
print("模拟退出/失效记录：")
synthetic_cases=synthetic_features.join(synthetic_states)
display(synthetic_cases.loc[synthetic_cases.reason.isin(["z_stop","relationship_failed","max_holding_exit"]),    ["z","beta","half_life","coint_p","state","reason"]].head(12))
L2_EXPERIMENTS.append({"id":"synthetic_regime_break","kind":"synthetic_mechanism",    "seed":L2_CONFIG["seed"],"ar_phi":.90,"break_index":synthetic_break_at,    "known_half_life":float(synthetic_reference_half),"not_historical_returns":True})


## 02E 练习参考与最终验收

02.10按本次实际指标生成参考差异，逐个模型核验窗口、费用、账户损益和不可借分支。只有全部断言通过后才打印L2 FINAL PASS并标记completed_and_verified。

标准版课件完成与个人自主研究分开：选择一个对照，先记录假设、只改一个因素，再填写实际指标、机制解释与否决证据。不得把未过滤诊断当作合格协整策略，也不得在历史留出期重新选配对。

In [ ]:
# 02.10 练习参考解答、完整验收与最终运行记录
# 此处只读本次真实输出；不编造个人研究结论，不把开发期称样本外。
reference_rows=[]
for name in ["fixed","unhedged","frozen_model","single_z","equal_weight"]:
    r=L2_SUMMARY.loc[name]; b=L2_SUMMARY.loc["rolling"]
    reference_rows.append({"comparison":name,"delta_total_return_vs_rolling":r.total_return-b.total_return,        "delta_max_drawdown_vs_rolling":r.max_drawdown-b.max_drawdown,        "mean_gross":r.mean_gross,"mean_net":r.mean_net,        "transaction_and_carry_cost":r.trade_cost+r.borrow_cost+r.funding_cost})
print("练习1 参考：保持同一滚动信号，仅改变金额对冲比，读取fixed与rolling；冻结全部模型另读frozen_model。")
display(pd.DataFrame(reference_rows).set_index("comparison").style.format({    "delta_total_return_vs_rolling":"{:+.2%}","delta_max_drawdown_vs_rolling":"{:+.2%}",    "mean_gross":"{:.2%}","mean_net":"{:.2%}","transaction_and_carry_cost":"{:.2f}"}))
print("练习2 参考：过滤基线若无交易，成本情景全为零；活跃诊断应与相同未过滤信号no_coint_filter比较。")
print("未过滤对冲诊断的真实结果：")
display(L2_HEDGE_TABLE.style.format(l2_fmt))
print("练习3 参考：不得按全样本最小p值挑配对；多个候选/多次试验需登记、在开发期选择、在新数据验证。")
print("练习4 参考：半衰期是历史局部估计；即使回归，市场损益仍可能小于交易和借券成本。")
print("进阶：预先登记一个OU、Kalman或因子残差假设，先验证因果性和基线，不在留出期选择自由度。")
# 最终核验逐腿与账户损益完全相等，并要求运行中的所有模型同一窗口。
validation=[]
def l2_final_check(name,ok):
    assert bool(ok),name
    validation.append({"check":name,"status":"PASS"})
l2_final_check("signal available before execution",(l2_interface.available_at<l2_interface.execution_at).all())
l2_final_check("OLS window ends before signal session",(l2_features.fit_last_session<l2_features.index).all())
l2_final_check("model accounting checks ran",np.isclose(reference_ledger.equity_end.iloc[-1],reference_cash))
l2_final_check("manual two-leg calculation",np.isclose(manual_fee,4.) and np.allclose(manual_units,[32.,-24.]))
for name,x in {**L2_LEDGERS,**{"scenario_"+k:v for k,v in L2_COMPARISONS.items()},
               **{"diagnostic_"+k:v for k,v in L2_HEDGE_DIAGNOSTIC.items()}}.items():
    l2_final_check(name+": same time window",x.index.equals(l2_target.index))
    l2_final_check(name+": PnL conservation",np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost-x.borrow_cost-x.funding_cost))
    l2_final_check(name+": cost signs",(x[["transaction_cost","borrow_cost","funding_cost"]]>=0).all().all())
    l2_final_check(name+": compound wealth",np.allclose((1+x.net_return).cumprod(),x.wealth))
    blocked=x.loc[x.blocked_by_borrow]
    l2_final_check(name+": unavailable closes portfolio",blocked.empty or np.allclose(blocked.gross_exposure,0))
L2_VALIDATION=pd.DataFrame(validation)
display(L2_VALIDATION)
L2_LOG.update(status="completed_and_verified",finished_utc=datetime.now(timezone.utc).isoformat(),    validation=L2_VALIDATION.to_dict("records"),data_manifest=L2_MANIFEST,    actual_execution_start=str(l2_target.index[0].date()),    actual_final_valuation=str(L2_LEDGERS["rolling"].end_session.iloc[-1].date()),    models=list(L2_LEDGERS),diagnostic_models=list(L2_HEDGE_DIAGNOSTIC),
    experiments=L2_EXPERIMENTS,configuration=L2_CONFIG.copy(),    personal_exercise_status="template and reference supplied; learner experiment remains independent",    limitations=["fixed ETF pair","historical borrow availability not observed",        "adjusted synthetic units","development data only","no taxes/market impact/real margin calls"])
print("L2 FINAL PASS：本课代码、手算、对照、结构变化示例、图表与运行记录已完成。")
print(json.dumps(L2_LOG,ensure_ascii=False,indent=2,default=str))
# 所有产物均保留在本Colab笔记本与输出；不导出本地文件。

# 简明验收卡：完整台账在上方，不需要下载额外文件。
from IPython.display import Markdown
display(Markdown("### 第二课课件验收完成"))
completion_card=pd.DataFrame({"项目":["02.1至02.10代码","时间因果与共同窗口","逐腿费用与损益守恒",
    "双腿手算及周末借券","对冲/成本/不可借诊断","结构变化示例","运行日志"],
    "状态":["已顺序执行","PASS","PASS","PASS","已完成并标注过滤消融","已完成并标注合成数据","completed_and_verified"]})
display(completion_card)
print(f"验收断言{len(L2_VALIDATION)}项全部通过；实际开发数据{L2_LOG['actual_execution_start']}至{L2_LOG['actual_final_valuation']}。")
